In [18]:
#Loading modules
import pandas as pd
import duckdb as db
import numpy as np
import scipy.stats as stats

In [19]:
#loading abbattoir lesions and contact farms dataframe

adf = db.query("SELECT * FROM './original_data/adf.parquet'").to_df()

adf

,vig_id,vig_date,result_date,bovine_id,farm_id,result,contact_order,last_contact,exposure_duration,herd_size
0,2622021,2021-10-15 13:16:22.0000000,2021-11-16,B1,F1,True,2,2013-03-10 00:00:00.000,575,3044.360000
1,1442022,2022-06-08 08:57:34.0000000,2022-06-23,B2,F2,False,3,2012-04-18 00:00:00.000,3529,6.350524
2,3712023,2023-07-14 12:26:07.0000000,2023-07-26,B3,F3,True,5,2022-09-02 00:00:00.000,1321,92.512491
3,3712023,2023-07-14 12:26:07.0000000,2023-07-26,B3,F4,True,4,2022-11-04 00:00:00.000,55,12.000000
4,4972024,2024-03-11 09:12:45.0000000,2024-03-20,B4,F5,True,2,2012-04-21 00:00:00.000,1273,39.860173
...,...,...,...,...,...,...,...,...,...,...
2291,7152025,2025-02-26 13:11:26.0000000,2025-03-05,B730,F554,False,2,2025-02-02 00:00:00.000,47,492.212766
2292,7122025,2025-02-24 10:22:59.0000000,2025-03-05,B436,F1845,True,3,2025-02-18 00:00:00.000,2,658.000000
2293,7122025,2025-02-24 10:22:59.0000000,2025-03-05,B436,F1846,True,6,2025-01-27 00:00:00.000,7,680.428571
2294,6912025,2025-02-11 14:12:15.0000000,2025-03-01,B732,F1847,True,3,2016-08-12 00:00:00.000,1092,168.654762


In [20]:
print(adf.vig_date.min(), adf.vig_date.max())

2021-01-06 14:17:33.0000000 2025-12-24 09:33:06.0000000


In [21]:
# Adding the route size as a variable route_size

rdf = db.query("""
            SELECT
                a.bovine_id,
                COUNT(a.farm_id) as route_size
            FROM adf a
            GROUP BY
            a.bovine_id
""").df()

adf = db.query("""
                SELECT
                    adf.*,
                    rdf.route_size

                FROM adf
                    JOIN rdf on rdf.bovine_id = adf.bovine_id
               """).df()
adf

,vig_id,vig_date,result_date,bovine_id,farm_id,result,contact_order,last_contact,exposure_duration,herd_size,route_size
0,2622021,2021-10-15 13:16:22.0000000,2021-11-16,B1,F1,True,2,2013-03-10 00:00:00.000,575,3044.360000,2
1,1442022,2022-06-08 08:57:34.0000000,2022-06-23,B2,F2,False,3,2012-04-18 00:00:00.000,3529,6.350524,4
2,3712023,2023-07-14 12:26:07.0000000,2023-07-26,B3,F3,True,5,2022-09-02 00:00:00.000,1321,92.512491,7
3,3712023,2023-07-14 12:26:07.0000000,2023-07-26,B3,F4,True,4,2022-11-04 00:00:00.000,55,12.000000,7
4,4972024,2024-03-11 09:12:45.0000000,2024-03-20,B4,F5,True,2,2012-04-21 00:00:00.000,1273,39.860173,3
...,...,...,...,...,...,...,...,...,...,...,...
2291,7152025,2025-02-26 13:11:26.0000000,2025-03-05,B730,F554,False,2,2025-02-02 00:00:00.000,47,492.212766,9
2292,7122025,2025-02-24 10:22:59.0000000,2025-03-05,B436,F1845,True,3,2025-02-18 00:00:00.000,2,658.000000,7
2293,7122025,2025-02-24 10:22:59.0000000,2025-03-05,B436,F1846,True,6,2025-01-27 00:00:00.000,7,680.428571,7
2294,6912025,2025-02-11 14:12:15.0000000,2025-03-01,B732,F1847,True,3,2016-08-12 00:00:00.000,1092,168.654762,4


In [22]:
# Filtering only the confirmed positive lesions

cdf = adf.query('result == True')

cdf

,vig_id,vig_date,result_date,bovine_id,farm_id,result,contact_order,last_contact,exposure_duration,herd_size,route_size
0,2622021,2021-10-15 13:16:22.0000000,2021-11-16,B1,F1,True,2,2013-03-10 00:00:00.000,575,3044.360000,2
2,3712023,2023-07-14 12:26:07.0000000,2023-07-26,B3,F3,True,5,2022-09-02 00:00:00.000,1321,92.512491,7
3,3712023,2023-07-14 12:26:07.0000000,2023-07-26,B3,F4,True,4,2022-11-04 00:00:00.000,55,12.000000,7
4,4972024,2024-03-11 09:12:45.0000000,2024-03-20,B4,F5,True,2,2012-04-21 00:00:00.000,1273,39.860173,3
6,2162022,2022-09-20 11:57:11.0000000,2022-09-29,B6,F7,True,1,2022-08-05 00:00:00.000,45,2073.311111,4
...,...,...,...,...,...,...,...,...,...,...,...
2285,6992025,2025-02-14 11:10:09.0000000,2025-03-01,B432,F1840,True,3,2021-01-25 00:00:00.000,795,101.213836,5
2292,7122025,2025-02-24 10:22:59.0000000,2025-03-05,B436,F1845,True,3,2025-02-18 00:00:00.000,2,658.000000,7
2293,7122025,2025-02-24 10:22:59.0000000,2025-03-05,B436,F1846,True,6,2025-01-27 00:00:00.000,7,680.428571,7
2294,6912025,2025-02-11 14:12:15.0000000,2025-03-01,B732,F1847,True,3,2016-08-12 00:00:00.000,1092,168.654762,4


In [23]:
# Creating PERT distribution (Vose, D. (2008)) with lambda values of Barlow, N. D., et al. (1997)

## Barlow, N. D., et al. (1997)
lambda_min = 1.0e-5  
lambda_mode = 2.7e-5  
lambda_max = 8.0e-5 

## Number of iterations
nit = 5000
## function to generate the distribution
def gen_pert_df(min_val, mode_val, max_val, size, seed=42):
    # Initialize the generator with a fixed seed
    rng = np.random.default_rng(seed)
    
    range_val = max_val - min_val
    if range_val == 0:
        samples = np.full(size, min_val)
    else:
        alpha = 1 + 4 * (mode_val - min_val) / range_val
        beta = 1 + 4 * (max_val - mode_val) / range_val
        
        
        samples = min_val + rng.beta(alpha, beta, size) * range_val
    
    return pd.DataFrame({
        'iteration_id': np.arange(size),
        'lambda_val': samples
    })

## Generate our pool of nit lambdas
pert_df = gen_pert_df(lambda_min, lambda_mode, lambda_max, nit,42)



In [24]:
# Applying the Exponential Dose-Response Model using 5000 Monte Carlo Iterations and ranking

db.query("""
        SELECT 
            l.iteration_id,
            c.bovine_id,
            c.farm_id,
            c.route_size,
            CASE 
            WHEN c.herd_size <= 1  THEN 0 
            ELSE 1 - exp(-1 * l.lambda_val * c.exposure_duration * ln(c.herd_size)) 
        END AS p_ij
        FROM cdf AS c
        CROSS JOIN pert_df AS l
""").to_parquet('mc_result.parquet')

In [26]:
# Coefficient of variation
cv_summary = db.query("""
    WITH per_contact_stats AS (
        SELECT 
            (STDDEV(p_ij) / NULLIF(AVG(p_ij), 0)) as cv
        FROM 'mc_result.parquet'
        GROUP BY bovine_id, farm_id
    )
    SELECT 
        count(*) as total_contacts,
        avg(cv) as mean_cv,
        stddev(cv) as std_cv,
        min(cv) as min_cv,
        quantile_cont(cv, 0.25) as q25_cv,
        quantile_cont(cv, 0.50) as median_cv,
        quantile_cont(cv, 0.75) as q75_cv,
        max(cv) as max_cv
    FROM per_contact_stats
""").df()

display(cv_summary)

,total_contacts,mean_cv,std_cv,min_cv,q25_cv,median_cv,q75_cv,max_cv
0,1452,0.347429,0.030062,0.18356,0.332358,0.359215,0.370422,0.373551


In [ ]:
# Table 2
table_2_df = db.query("""
    WITH per_contact_stats AS (
        SELECT 
            (STDDEV(p_ij) / NULLIF(AVG(p_ij), 0)) as cv,
            AVG(p_ij) as mean_p
        FROM 'mc_result.parquet'
        GROUP BY bovine_id, farm_id
    )
    SELECT 
        COUNT(cv) as "Active Contacts (n)",
        round(avg(cv), 3) as "Mean CV",
        round(stddev(cv), 3) as "SD",
        round(min(cv), 3) as "Min",
        round(quantile_cont(cv, 0.25), 3) as "Q1",
        round(quantile_cont(cv, 0.50), 3) as "Median",
        round(quantile_cont(cv, 0.75), 3) as "Q3",
        round(max(cv), 3) as "Max"
    FROM per_contact_stats
""").df()

display(table_1_df)

,Active Contacts (n),Mean CV,SD,Min,Q1,Median,Q3,Max
0,1301,0.347,0.03,0.184,0.332,0.359,0.37,0.374


In [10]:
# Final model result incorporating deterministic and simulation results

## Barlow, N. D., et al. (1997)
lambda_min = 1.0e-5  
lambda_mode = 2.7e-5  
lambda_max = 8.0e-5 

# 2. Calculate the PERT Mean
deterministic_lambda = (lambda_min + 4 * lambda_mode + lambda_max) / 6
print(f"Calculated PERT Mean Lambda: {deterministic_lambda:.4e}")

# 3. Build the final results featuring stochastic metrics vs. deterministic P
model_result = db.query(f"""
WITH stochastic_stats AS (
    SELECT 
        m.bovine_id,
        m.farm_id,
        cdf.vig_date,
        cdf.contact_order,
        cdf.last_contact,
        cdf.exposure_duration,
        cdf.herd_size,
        cdf.route_size,
        AVG(m.p_ij) as p_mean_stochastic,
        STDDEV(m.p_ij) as p_std,
        MIN(m.p_ij) as p_min,
        MAX(m.p_ij) as p_max,
        QUANTILE_CONT(m.p_ij, 0.025) as p_ci_lower,
        QUANTILE_CONT(m.p_ij, 0.975) as p_ci_upper,
        (STDDEV(m.p_ij) / NULLIF(AVG(m.p_ij), 0)) as cv
    FROM 'mc_result.parquet' m
        JOIN cdf on cdf.bovine_id = m.bovine_id
            AND m.farm_id = cdf.farm_id
    GROUP BY 1, 2, 3,4,5,6,7,8
)
SELECT 
    s.*,
    CASE 
        WHEN c.herd_size <= 1 THEN 0
        ELSE 1 - exp(-1 * {deterministic_lambda} * c.exposure_duration * ln(c.herd_size + 1)) 
        END AS p_deterministic
FROM stochastic_stats s
JOIN cdf c ON s.bovine_id = c.bovine_id AND s.farm_id = c.farm_id
"""

).df()



Calculated PERT Mean Lambda: 3.3000e-05


In [11]:
db.query("SELECT * FROM model_result").to_parquet("model_result.parquet")
model_result

,bovine_id,farm_id,vig_date,contact_order,last_contact,exposure_duration,herd_size,route_size,p_mean_stochastic,p_std,p_min,p_max,p_ci_lower,p_ci_upper,cv,p_deterministic
0,B347,F439,2022-08-17 13:53:09.0000000,1,2021-04-19 00:00:00.000,1617,217.469021,5,0.244424,0.078709,0.084779,0.477551,0.112291,0.401474,0.322019,0.249816
1,B348,F95,2024-01-02 10:20:25.0000000,1,2021-11-24 00:00:00.000,768,40.757216,3,0.088742,0.031576,0.028570,0.191376,0.038222,0.154597,0.355821,0.090246
2,B447,F1302,2023-07-19 12:18:48.0000000,1,2023-07-17 00:00:00.000,1,11043.000000,3,0.000306,0.000114,0.000095,0.000694,0.000127,0.000549,0.373510,0.000307
3,B651,F16,2022-10-18 16:54:52.0000000,1,2022-10-14 00:00:00.000,1,32594.000000,3,0.000341,0.000127,0.000106,0.000775,0.000142,0.000613,0.373503,0.000343
4,B695,F1079,2021-12-08 13:26:02.0000000,2,2016-10-19 00:00:00.000,72,1909.875000,3,0.017687,0.006546,0.005522,0.039766,0.007417,0.031572,0.370110,0.017791
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1447,B558,F1588,2025-05-08 11:27:18.0000000,2,2025-04-28 00:00:00.000,6,374.000000,3,0.001167,0.000436,0.000362,0.002648,0.000486,0.002094,0.373342,0.001173
1448,B558,F940,2025-05-08 11:27:18.0000000,1,2025-05-04 00:00:00.000,48,15742.500000,3,0.015106,0.005599,0.004711,0.034013,0.006329,0.026988,0.370617,0.015191
1449,B500,F1596,2021-07-27 16:16:05.0000000,4,2012-01-01 00:00:00.000,2950,23.228475,5,0.258025,0.082279,0.090130,0.499528,0.119262,0.421468,0.318880,0.266777
1450,B36,F248,2024-09-09 15:59:13.0000000,1,2016-04-26 00:00:00.000,3223,207.669563,7,0.419255,0.116901,0.160597,0.722783,0.209727,0.637348,0.278831,0.433363
